In [7]:
!pip install duckdb openpyxl
import duckdb
import pandas as pd

# 1. Connect to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

# 2. Load Excel files using raw strings (r"...") to prevent path backslash errors
chart_of_accounts = pd.read_excel(r"C:\Users\hasan\Downloads\Chart of Accounts.xlsx", sheet_name='Income Statement')
income_statement = pd.read_excel(r"C:\Users\hasan\Downloads\Income Statement.xlsx")
budget = pd.read_excel(r"C:\Users\hasan\Downloads\Budget.xlsx")

# 3. Register DataFrames as DuckDB views
con.register('coa', chart_of_accounts)
con.register('actuals', income_statement)
con.register('budget', budget)

# 4. Query 1: 2018 Entity P&L Performance Summary
query_entity_performance = """
SELECT 
    a.Entity,
    ROUND(SUM(CASE WHEN c.Category = 'Revenue' THEN a.Amount ELSE 0 END), 2) AS Total_Revenue,
    ROUND(SUM(CASE WHEN c.Category = 'Cost of Revenue' THEN a.Amount ELSE 0 END), 2) AS Total_COGS,
    ROUND(SUM(CASE WHEN c.Category = 'Expense' THEN a.Amount ELSE 0 END), 2) AS Total_OpEx,
    ROUND(
        SUM(CASE WHEN c.Category = 'Revenue' THEN a.Amount ELSE 0 END) - 
        SUM(CASE WHEN c.Category = 'Cost of Revenue' THEN a.Amount ELSE 0 END) - 
        SUM(CASE WHEN c.Category = 'Expense' THEN a.Amount ELSE 0 END), 2
    ) AS Net_Income
FROM actuals a
JOIN coa c 
    ON a.Account = c.Account 
   AND a.Entity = c.Entity
WHERE strftime(a.Date, '%Y') = '2018'
GROUP BY a.Entity
ORDER BY Net_Income DESC;
"""

df_entity_perf = con.execute(query_entity_performance).df()
print("--- 2018 Entity Performance Summary ---")
display(df_entity_perf)

# 5. Query 2: Budget vs. Actuals (BvA) Variance Analysis
query_bva = """
WITH actuals_grouped AS (
    SELECT 
        a.Entity,
        strftime(a.Date, '%Y-%m') AS Month,
        TRIM(c."Group") AS Account_Group,
        SUM(a.Amount) AS Actual_Amount
    FROM actuals a
    JOIN coa c 
        ON a.Account = c.Account 
       AND a.Entity = c.Entity
    WHERE strftime(a.Date, '%Y') = '2018'
    GROUP BY a.Entity, strftime(a.Date, '%Y-%m'), TRIM(c."Group")
),
budget_grouped AS (
    SELECT 
        Entity,
        strftime(Date, '%Y-%m') AS Month,
        TRIM("Group") AS Account_Group,
        SUM(Budget) AS Budget_Amount
    FROM budget
    WHERE strftime(Date, '%Y') = '2018'
    GROUP BY Entity, strftime(Date, '%Y-%m'), TRIM("Group")
)
SELECT 
    COALESCE(a.Entity, b.Entity) AS Entity,
    COALESCE(a.Month, b.Month) AS Month,
    COALESCE(a.Account_Group, b.Account_Group) AS Account_Group,
    ROUND(COALESCE(b.Budget_Amount, 0), 2) AS Budgeted,
    ROUND(COALESCE(a.Actual_Amount, 0), 2) AS Actual,
    ROUND(COALESCE(a.Actual_Amount, 0) - COALESCE(b.Budget_Amount, 0), 2) AS Variance_USD,
    ROUND(
        CASE 
            WHEN COALESCE(b.Budget_Amount, 0) = 0 THEN NULL
            ELSE ((COALESCE(a.Actual_Amount, 0) - COALESCE(b.Budget_Amount, 0)) / b.Budget_Amount) * 100 
        END, 2
    ) AS Variance_Pct
FROM actuals_grouped a
FULL OUTER JOIN budget_grouped b 
    ON a.Entity = b.Entity 
   AND a.Month = b.Month 
   AND a.Account_Group = b.Account_Group
ORDER BY Entity, Month, Account_Group;
"""

df_bva = con.execute(query_bva).df()
con.register('df_bva', df_bva)

print("\n--- Budget vs. Actuals Sample (First 10 Rows) ---")
display(df_bva.head(10))

# 6. Query 3: Identifying High Operational Expense Overruns (> 15% Over Budget)
query_overruns = """
SELECT 
    Entity,
    Account_Group,
    SUM(Budgeted) AS Total_Budget,
    SUM(Actual) AS Total_Actual,
    SUM(Variance_USD) AS Total_Variance_USD,
    ROUND((SUM(Actual) - SUM(Budgeted)) / SUM(Budgeted) * 100, 2) AS Total_Variance_Pct
FROM df_bva
WHERE Account_Group NOT IN ('Revenue', 'Other Income')
GROUP BY Entity, Account_Group
HAVING SUM(Budgeted) > 0 AND (SUM(Actual) - SUM(Budgeted)) / SUM(Budgeted) > 0.15
ORDER BY Total_Variance_USD DESC;
"""

df_overruns = con.execute(query_overruns).df()
print("\n--- Operational Overruns (>15% Variance) ---")
display(df_overruns)

--- 2018 Entity Performance Summary ---


,Entity,Total_Revenue,Total_COGS,Total_OpEx,Net_Income
0,Entity 1,3823942.90,2210117.52,1057827.27,555998.11
1,Entity 6,68804.34,60781.81,0.00,8022.53
2,Entity 4,0.00,0.00,17945.74,-17945.74
3,Entity 5,1091950.54,746679.51,518839.08,-173568.05
4,Entity 2,63237.45,57385.73,203499.90,-197648.18
5,Entity 3,0.00,0.00,448058.94,-448058.94



--- Budget vs. Actuals Sample (First 10 Rows) ---


,Entity,Month,Account_Group,Budgeted,Actual,Variance_USD,Variance_Pct
0,Entity 1,2018-01,-Advertising,26677.00,3118.94,-23558.06,-88.31
1,Entity 1,2018-01,-Back Office Expenses,63859.66,0.00,-63859.66,-100.00
2,Entity 1,2018-01,-Depreciation Expense,21066.86,27766.79,6699.93,31.80
3,Entity 1,2018-01,-Other Expenses,39712.00,484545.97,444833.97,1120.15
4,Entity 1,2018-01,-Trade License / SSHIP,0.00,0.00,0.00,NaN
5,Entity 1,2018-01,Communication,1636.22,2172.49,536.27,32.78
6,Entity 1,2018-01,Cost of Revenue,1147111.00,727414.43,-419696.57,-36.59
7,Entity 1,2018-01,Finance Costs,21000.00,0.00,-21000.00,-100.00
8,Entity 1,2018-01,Financial Costs,551.22,0.00,-551.22,-100.00
9,Entity 1,2018-01,Insurance Expenses,6464.45,6297.17,-167.28,-2.59



--- Operational Overruns (>15% Variance) ---


,Entity,Account_Group,Total_Budget,Total_Actual,Total_Variance_USD,Total_Variance_Pct
0,Entity 5,-Other Expenses,216692.3,274932.73,58240.44,26.88
